# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hafsaShaban/flyrank_internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os
if not os.path.exists('/content/flyrank_internship'):
    !git clone https://github.com/hafsaShaban/flyrank_internship.git
%cd /content/flyrank_internship

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print("Loaded:", len(df), "rows")

/content/flyrank_internship
Loaded: 30000 rows


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Building the final ranked queue using the model from ML-08/ML-09, rebuilt here for a clean end-to-end run. Claim ladder rung used: "a validated model that ranks out-of-sample" → "the model ranks pages at precision@50 of [X], on a client-holdout split" — never "will decline" or "should be fixed," per the banned-phrasing lis

In [3]:
numeric_features = ['word_count','char_count','ctr','avg_position','engagement_rate','scroll_rate',
    'ai_traffic_pct','content_age_days','days_since_last_update','search_volume','cpc',
    'impressions_90d','clicks_90d','pageviews_90d','sessions_90d','users_90d',
    'engaged_sessions_90d','ai_sessions_90d','scroll_events_90d',
    'days_with_impressions','days_with_sessions']
categorical_features = ['content_type','main_intent','competition_level','freshness_tier',
    'word_count_tier','char_count_tier','impression_tier','position_tier']

X = df[numeric_features].copy()
for col in numeric_features:
    X[f'has_{col}'] = X[col].notnull().astype(int)
    X[col] = X[col].fillna(0)
X_cat = pd.get_dummies(df[categorical_features], dummy_na=True)
X = pd.concat([X, X_cat], axis=1)
y = df['is_declining_label']
groups = df['client_id']

gkf = GroupKFold(n_splits=5)
scores_out = np.zeros(len(y))
for train_idx, test_idx in gkf.split(X, y, groups):
    m = RandomForestClassifier(n_estimators=200, random_state=42)
    m.fit(X.iloc[train_idx], y.iloc[train_idx])
    scores_out[test_idx] = m.predict_proba(X.iloc[test_idx])[:, 1]

df['model_score'] = scores_out

def reason_code(row):
    codes = []
    if row['days_since_last_update'] >= 180 and row['impressions_90d'] >= 500:
        codes.append('stale_but_visible')
    if row['ctr'] < df['ctr'].median() and row['impressions_90d'] >= 500:
        codes.append('low_ctr_visible_page')
    if row['model_score'] >= 0.65:
        codes.append('model_decline_risk')
    return ','.join(codes) if codes else 'no_flag'

df['reason_code'] = df.apply(reason_code, axis=1)
queue = df[['content_id','client_id','model_score','reason_code','impressions_90d','ctr',
            'days_since_last_update','is_declining_label']].sort_values('model_score', ascending=False)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

p50 = precision_at_k(df['model_score'], df['is_declining_label'], 50)
print(f"Model ranks pages at precision@50 of {p50:.3f}, client-holdout split, base rate {y.mean():.3f}")
queue.head(10)


Model ranks pages at precision@50 of 0.580, client-holdout split, base rate 0.542


,content_id,client_id,model_score,reason_code,impressions_90d,ctr,days_since_last_update,is_declining_label
17451,content_d015eb800625,client_d029fa3a95,0.975,"low_ctr_visible_page,model_decline_risk",1064,0.00,20,0
19421,content_102d1a1e54a6,client_f369cb89fc,0.965,model_decline_risk,1183,0.08,8,1
9750,content_3537771c4049,client_bbb965ab0c,0.960,"low_ctr_visible_page,model_decline_risk",1159,0.00,15,1
25439,content_3d8f6737ad9d,client_a88a7902cb,0.960,model_decline_risk,152,0.00,8,1
22516,content_db74537c5636,client_8527a891e2,0.960,model_decline_risk,171,0.00,102,1
24167,content_0895470266fd,client_6208ef0f77,0.960,"low_ctr_visible_page,model_decline_risk",709,0.00,20,1
15399,content_f02bbe6f2411,client_d029fa3a95,0.955,model_decline_risk,86,0.00,20,1
16186,content_fd2075ea5f6a,client_d029fa3a95,0.950,model_decline_risk,1884,0.11,20,1
14343,content_9ac61c04930e,client_8527a891e2,0.950,model_decline_risk,1828,0.22,104,1
28582,content_f49660e074e9,client_8527a891e2,0.950,model_decline_risk,1432,0.35,102,0


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

"Who uses this": a content reviewer with limited weekly capacity, as a starting point for their own judgment — not an automated action list. Claim ladder check: this is decision-support only — "these pages look worth reviewing first, because [reason code]," never "these pages will decline" or "refreshing these will improve rankings." Limits: the label is a current-window proxy, not an observed future outcome (per the flyrank-data skill); this is cross-sectional data with no intervention, so it cannot support "doing X will produce Y."

In [4]:
print("Queue built from cross-sectional starter data — no intervention was run.")
print("Any causal language about refresh outcomes is NOT supported by this data.")


Queue built from cross-sectional starter data — no intervention was run.
Any causal language about refresh outcomes is NOT supported by this data.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

What a person must check before acting: verify the page manually (not just trust the score), confirm the reason code still applies, check for consolidation/seasonality look-alikes (per the Lane Guide's decline vs. consolidation/seasonality/noise table) before assuming real decline. No-go list — never automate: publishing changes without human sign-off, treating a high score as proof of causation, using client-identifying details in any output.

In [5]:
top20 = queue.head(20)
weak_picks = top20[top20['is_declining_label'] == 0]
print("Weak picks a human reviewer should catch (flagged high, but not actually declining):")
print(weak_picks[['content_id','model_score','reason_code']])


Weak picks a human reviewer should catch (flagged high, but not actually declining):
                 content_id  model_score  \
17451  content_d015eb800625        0.975   
28582  content_f49660e074e9        0.950   
22042  content_2ba626fea4d6        0.950   
1559   content_c0ba9d88864f        0.950   
23501  content_e49e368bdeb2        0.945   
22867  content_e0127bfa31e5        0.940   
18999  content_805132378465        0.935   

                                   reason_code  
17451  low_ctr_visible_page,model_decline_risk  
28582                       model_decline_risk  
22042                       model_decline_risk  
1559                        model_decline_risk  
23501                       model_decline_risk  
22867                       model_decline_risk  
18999                       model_decline_risk  


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

What would signal these recommendations went stale: precision@K on a fresh batch dropping notably below the current 0.[X]; the declining-label base rate shifting significantly from the current baseline; a new content type or client entering the data that wasn't represented in training; reason-code distributions shifting sharply, which could mean underlying signals drifted.

In [6]:
print("Current base rate to monitor against:", y.mean())
print("Current precision@50 to monitor against:", p50)
print("Reason code distribution to monitor for drift:")
print(df['reason_code'].value_counts(normalize=True))


Current base rate to monitor against: 0.5420666666666667
Current precision@50 to monitor against: 0.58
Reason code distribution to monitor for drift:
reason_code
no_flag                                                      0.623067
model_decline_risk                                           0.238600
low_ctr_visible_page                                         0.075067
low_ctr_visible_page,model_decline_risk                      0.062700
stale_but_visible                                            0.000233
stale_but_visible,model_decline_risk                         0.000233
stale_but_visible,low_ctr_visible_page,model_decline_risk    0.000100
Name: proportion, dtype: float64


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [7]:
import os
os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/action_playbook_queue.csv', index=False)
print("Saved", len(queue), "rows to work/outputs/action_playbook_queue.csv")


Saved 30000 rows to work/outputs/action_playbook_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.